# multi_strat_049

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (74).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `intraday` |
| Timeframe | `unknown` |
| Code cells | 7 |
| Detected functions | process_ticker, backtest_trades, generate_signals, process_ticker, backtest_trades, grid_search_params, evaluate_params |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/5min_data_1.csv')
data

In [ ]:
data["Ticker"].unique()

In [ ]:
import pandas as pd

# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2015-01-01'
end_date = '2024-12-31'

# Define the tickers you want to filter
selected_tickers = ['ADANIENT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

data["signal"] = 0  # Default value

# Long Entry Condition
data.loc[(data["SAMX"] <= 50) & (data["PVS"] >= 3) & (data["VMS"] >= 1), "signal"] = 1

# Short Entry Condition (More Strict)
data.loc[(data["SAMX"] >= -50) & (data["PVS"] <= -3) & (data["VMS"] >= 1), "signal"] = 2

# Function to process trades for a single ticker
def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time):
    """Processes trades for a single ticker, applying entry/exit logic and calculating PnL."""
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["signal"]

        # Check for open positions and apply exit logic
        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            # Compute Take Profit (TP) & Stop Loss (SL) prices
            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            # Determine exit conditions
            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif timestamp.astype('datetime64[m]').astype(str)[-5:] >= end_time:
                exit_price, exit_reason = close, "EOD Close"

            if exit_price is not None:
                # Apply slippage
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)

                # Calculate PnL including brokerage
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl

                # Store trade details
                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        # Open new position
        if ticker not in open_positions and timestamp.astype('datetime64[m]').astype(str)[-5:] < stop_trading_time:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

# Main backtesting function
def backtest_trades(data):
    """Runs backtest for all tickers in the dataset using parallel processing."""

    # Trading parameters
    initial_capital = 100000  # ₹1 lakh per trade
    brokerage = 0.001  # 0.1% brokerage
    slippage = 0.0001  # 0.01% slippage
    tp_pct = 0.02  # 1.5% take-profit
    sl_pct = 0.005  # 0.5% stop-loss
    end_time = "15:25"
    stop_trading_time = "15:15"

    # Get unique tickers
    unique_tickers = data['Ticker'].unique()

    # Convert DataFrame to NumPy structured array for performance
    dtype = [
        ("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"),
        ("PVS", "f8"), ("VMS", "f8"), ("SAMX", "f8"), ("signal", "i4")
    ]

    data_np = np.array(
        list(data[["Ticker", "Date", "close", "open", "high", "low", "PVS", "VMS", "SAMX", "signal"]]
            .itertuples(index=False, name=None)),
        dtype=dtype
    )

    # Run backtest in parallel
    results = Parallel(n_jobs=-1)(delayed(process_ticker)(ticker, data_np[data_np['Ticker'] == ticker], initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time) for ticker in tqdm(unique_tickers))

    # Flatten results into DataFrame
    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

# Example usage
# data = pd.read_csv("your_data.csv")  # Load dataset
data['Date'] = pd.to_datetime(data['Date'])  # Ensure Date column is in datetime format
tradebook = backtest_trades(data)
tradebook.to_csv("/content/drive/MyDrive/tradebook_adanient.csv", index=False)
tradebook

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
from itertools import product
import sys

# Function to generate signals based on parameters
def generate_signals(data, long_samx, long_pvs, long_vms, short_samx, short_pvs, short_vms):
    """Generates trading signals based on given SAMX, PVS, VMS thresholds."""
    data = data.copy()
    data["signal"] = 0

    # Long Entry Condition
    data.loc[(data["SAMX"] <= long_samx) & (data["PVS"] >= long_pvs) & (data["VMS"] >= long_vms), "signal"] = 1

    # Short Entry Condition
    data.loc[(data["SAMX"] >= short_samx) & (data["PVS"] <= short_pvs) & (data["VMS"] >= short_vms), "signal"] = 2

    return data

# Function to process trades for a single ticker
def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time):
    """Processes trades for a single ticker, applying entry/exit logic and calculating PnL."""
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["signal"]

        # Validate prices
        if any(p <= 0 for p in [close, open_price, high, low]):
            continue

        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif timestamp.astype('datetime64[m]').astype(str)[-5:] >= end_time:
                exit_price, exit_reason = close, "EOD Close"

            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl
                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        if ticker not in open_positions and timestamp.astype('datetime64[m]').astype(str)[-5:] < stop_trading_time:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

# Backtesting function
def backtest_trades(data, long_samx, long_pvs, long_vms, short_samx, short_pvs, short_vms):
    """Runs backtest for all tickers with given parameters."""
    initial_capital = 100000
    brokerage = 0.001
    slippage = 0.0001
    tp_pct = 0.02  # Reduced to 1% for intraday
    sl_pct = 0.05  # Reduced to 0.25% for intraday
    end_time = "15:25"
    stop_trading_time = "15:25"  # Extended to allow late signals

    data = generate_signals(data, long_samx, long_pvs, long_vms, short_samx, short_pvs, short_vms)

    # Log signal counts
    signal_counts = data['signal'].value_counts()
    signal_info = (f"Signals: Long={signal_counts.get(1, 0)}, Short={signal_counts.get(2, 0)}, "
                   f"No Signal={signal_counts.get(0, 0)}")

    unique_tickers = data['Ticker'].unique()

    dtype = [
        ("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"),
        ("PVS", "f8"), ("VMS", "f8"), ("SAMX", "f8"), ("signal", "i4")
    ]

    try:
        data_np = np.array(
            list(data[["Ticker", "Date", "close", "open", "high", "low", "PVS", "VMS", "SAMX", "signal"]]
                .itertuples(index=False, name=None)),
            dtype=dtype
        )
    except Exception as e:
        return pd.DataFrame(), 0, f"Data conversion error: {str(e)}"

    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(
            ticker, data_np[data_np['Ticker'] == ticker], initial_capital, brokerage, slippage,
            tp_pct, sl_pct, end_time, stop_trading_time
        ) for ticker in tqdm(unique_tickers, desc="Processing tickers")
    )

    trades = [trade for result in results for trade in result]
    tradebook = pd.DataFrame(trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
                                             "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

    total_pnl = tradebook["Cumulative PnL"].iloc[-1] if not tradebook.empty else 0
    return tradebook, total_pnl, signal_info

# Grid search function
def grid_search_params(data):
    """Performs grid search to find optimal SAMX, PVS, VMS thresholds."""
    data = data.copy()
    data['Date'] = pd.to_datetime(data['Date'])

    required_columns = ['Ticker', 'Date', 'close', 'open', 'high', 'low', 'PVS', 'VMS', 'SAMX']
    if not all(col in data.columns for col in required_columns):
        raise ValueError(f"DataFrame missing required columns: {required_columns}")

    # Print data diagnostics
    print("Data Diagnostics:")
    print(f"Columns: {data.columns.tolist()}")
    print(f"SAMX range: {data['SAMX'].min()} to {data['SAMX'].max()}")
    print(f"PVS range: {data['PVS'].min()} to {data['PVS'].max()}")
    print(f"VMS range: {data['VMS'].min()} to {data['VMS'].max()}")
    print(f"Date range: {data['Date'].min()} to {data['Date'].max()}")
    print(f"Number of rows: {len(data)}")
    print(f"Valid prices: {not (data[['close', 'open', 'high', 'low']] <= 0).any().any()}")

    # Test signal generation
    test_params = (-70, 1, 0, -70, -4.5, 0)  # Adjusted to data ranges
    test_data = generate_signals(data, *test_params)
    signal_counts = test_data['signal'].value_counts()
    print("Signal Generation Test (long_samx=-70, long_pvs=1, long_vms=0, short_samx=-70, short_pvs=-4.5, short_vms=0):")
    print(f"Long signals (1): {signal_counts.get(1, 0)}")
    print(f"Short signals (2): {signal_counts.get(2, 0)}")
    print(f"No signals (0): {signal_counts.get(0, 0)}")

    # Define parameter grid aligned with data
    param_grid = {
        'long_samx': np.arange(0, 100, 2),  # -70, -65, -60, ..., -30
        'long_pvs': np.arange(0.1, 5, 0.2),     # 2, 2.5, 3, ..., 5
        'long_vms': np.arange(-2, 2, 0.1),     # 1, 1.5, 2
        'short_samx': np.arange(-100, 0, 2), # -70, -65, -60, ..., -30
        'short_pvs': np.arange(-0.1, -5, 0.2),  # -5, -4.5, ..., -2
        'short_vms': np.arange(-2, 2, 0.1)     # 1, 1.5, 2
    }

    param_combinations = list(product(
        param_grid['long_samx'], param_grid['long_pvs'], param_grid['long_vms'],
        param_grid['short_samx'], param_grid['short_pvs'], param_grid['short_vms']
    ))
    print(f"Total parameter combinations: {len(param_combinations)}")

    log_file = "/content/drive/MyDrive/grid_search_log.txt"
    open(log_file, "w").close()

    def evaluate_params(params):
        long_samx, long_pvs, long_vms, short_samx, short_pvs, short_vms = params
        try:
            tradebook, total_pnl, signal_info = backtest_trades(
                data, long_samx, long_pvs, long_vms, short_samx, short_pvs, short_vms
            )
        except Exception as e:
            error_msg = (f"Error for params: long_samx={long_samx}, long_pvs={long_pvs}, long_vms={long_vms}, "
                         f"short_samx={short_samx}, short_pvs={short_pvs}, short_vms={short_vms}, Error={str(e)}")
            print(error_msg, flush=True)
            with open(log_file, "a") as f:
                f.write(error_msg + "\n")
            return {
                'long_samx': long_samx, 'long_pvs': long_pvs, 'long_vms': long_vms,
                'short_samx': short_samx, 'short_pvs': short_pvs, 'short_vms': short_vms,
                'total_pnl': 0,
                'tradebook': pd.DataFrame()
            }
        result_msg = (f"Params: long_samx={long_samx}, long_pvs={long_pvs}, long_vms={long_vms}, "
                      f"short_samx={short_samx}, short_pvs={short_pvs}, short_vms={short_vms}, "
                      f"Trades={len(tradebook)}, {signal_info}, Cumulative PnL={total_pnl}")
        print(result_msg, flush=True)
        with open(log_file, "a") as f:
            f.write(result_msg + "\n")
        return {
            'long_samx': long_samx, 'long_pvs': long_pvs, 'long_vms': long_vms,
            'short_samx': short_samx, 'short_pvs': short_pvs, 'short_vms': short_vms,
            'total_pnl': total_pnl,
            'tradebook': tradebook
        }

    # Run grid search single-threaded for debugging
    results = Parallel(n_jobs=1)(
        delayed(evaluate_params)(params) for params in tqdm(param_combinations, desc="Grid Search")
    )

    results_df = pd.DataFrame([{
        'long_samx': r['long_samx'], 'long_pvs': r['long_pvs'], 'long_vms': r['long_vms'],
        'short_samx': r['short_samx'], 'short_pvs': r['short_pvs'], 'short_vms': r['short_vms'],
        'total_pnl': r['total_pnl']
    } for r in results if 'total_pnl' in r])

    if results_df.empty:
        print("No valid results found. All parameter combinations may have failed or produced no trades.")
        return None, pd.DataFrame(), results_df

    best_result = results_df.loc[results_df['total_pnl'].idxmax()]
    best_params = {
        'long_samx': best_result['long_samx'], 'long_pvs': best_result['long_pvs'], 'long_vms': best_result['long_vms'],
        'short_samx': best_result['short_samx'], 'short_pvs': best_result['short_pvs'], 'short_vms': best_result['short_vms']
    }

    best_tradebook = next((r['tradebook'] for r in results if all(
        r[f] == best_result[f] for f in ['long_samx', 'long_pvs', 'long_vms', 'short_samx', 'short_pvs', 'short_vms']
    )), pd.DataFrame())

    return best_params, best_tradebook, results_df

# Example usage
# data = pd.read_csv("your_data.csv")
best_params, best_tradebook, results_df = grid_search_params(data)
best_tradebook.to_csv("/content/drive/MyDrive/tradebook_optimized.csv", index=False)
results_df.to_csv("/content/drive/MyDrive/grid_search_results.csv", index=False)
print("Best Parameters:")
print(best_params)
print(f"Best Cumulative PnL: {best_tradebook['Cumulative PnL'].iloc[-1] if not best_tradebook.empty else 0}")

In [ ]:
param_grid = {
        'long_samx': np.arange(0, 100, 2),  # -70, -65, -60, ..., -30
        'long_pvs': np.arange(0.1, 5, 0.2),     # 2, 2.5, 3, ..., 5
        'long_vms': np.arange(-2, 2, 0.1),     # 1, 1.5, 2
        'short_samx': np.arange(-100, 0, 2), # -70, -65, -60, ..., -30
        'short_pvs': np.arange(-0.1, -5, 0.2),  # -5, -4.5, ..., -2
        'short_vms': np.arange(-2, 2, 0.1)     # 1, 1.5, 2
    }